# Прогнозирование энергопотребления зданий (BDGP2)

Этот ноутбук демонстрирует:
- Загрузку данных Building Data Genome Project 2
- Исследовательский анализ данных (EDA)
- Обучение модели прогнозирования (LinearRegression / RandomForest)
- Оценку метрик: MAE, RMSE, R²
- Визуализацию прогноза vs реальные данные

## Модуль предиктивной аналитики диплома

## 1. Импорт библиотек

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

# Настройка визуализации
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("✓ Библиотеки импортированы")

## 2. Определение путей к данным

In [ ]:
# Получение корневой директории проекта
PROJECT_ROOT = Path.cwd().parent

# Пути к данным
DATASETS_DIR = PROJECT_ROOT / 'datasets'
ENERGY_DIR = DATASETS_DIR / 'energy' / 'data'
MODELS_DIR = PROJECT_ROOT / 'models' / 'predictive'

# Пути к данным BDGP2
BDGP2_DIR = ENERGY_DIR / 'bdgp2'

print(f"📁 Корень проекта: {PROJECT_ROOT}")
print(f"📁 Данные энергопотребления: {ENERGY_DIR}")
print(f"📁 Модели: {MODELS_DIR}")

# Проверка наличия данных
if not ENERGY_DIR.exists():
    print("⚠️  Директория energy не найдена!")
    print("Запустите: python scripts/download_datasets.py")
else:
    print("✓ Директория energy найдена")
    
# Создание директории для моделей
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Загрузка данных BDGP2

In [ ]:
def load_bdgp2_data(data_dir: Path) -> pd.DataFrame:
    """
    Загрузить данные Building Data Genome Project 2.
    
    Returns:
        DataFrame с почасовыми данными энергопотребления.
    """
    # Поиск CSV файлов
    csv_files = list(data_dir.glob('*.csv'))
    
    if not csv_files:
        # Поиск в поддиректориях
        csv_files = list(data_dir.rglob('*.csv'))
    
    if not csv_files:
        print(f"⚠️  CSV файлы не найдены в {data_dir}")
        return pd.DataFrame()
    
    print(f"Найдено CSV файлов: {len(csv_files)}")
    
    # Загрузка первого файла (или объединение нескольких)
    df = pd.read_csv(csv_files[0], parse_dates=True)
    
    print(f"✓ Загружено данных: {len(df)} записей")
    print(f"  Колонки: {list(df.columns)[:10]}...")
    
    return df


# Загрузка данных
print("Загрузка данных BDGP2...\n")
energy_df = load_bdgp2_data(BDGP2_DIR)

if not energy_df.empty:
    print(f"\n📊 Размер данных: {energy_df.shape}")
    print(f"\nПервые 5 строк:")
    display(energy_df.head())

## 4. Исследовательский анализ данных (EDA)

In [ ]:
if not energy_df.empty:
    print("=" * 60)
    print("📈 ИССЛЕДОВАТЕЛЬСКИЙ АНАЛИЗ ДАННЫХ")
    print("=" * 60)
    
    # Основная статистика
    print("\n📊 Основная статистика:")
    display(energy_df.describe())
    
    # Проверка пропусков
    print("\n🔍 Пропущенные значения:")
    missing = energy_df.isnull().sum()
    missing_pct = (missing / len(energy_df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Пропуски': missing,
        '%': missing_pct
    }).sort_values('%', ascending=False)
    
    print(missing_df[missing_df['Пропуски'] > 0].head(10))
    
    # Визуализация распределения
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # График 1: Распределение целевой переменной
    numeric_cols = energy_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        target_col = numeric_cols[0]
        
        axes[0, 0].hist(energy_df[target_col].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[0, 0].set_xlabel('Значение')
        axes[0, 0].set_ylabel('Частота')
        axes[0, 0].set_title(f'Распределение: {target_col}')
        
        # График 2: Временной ряд
        if 'timestamp' in energy_df.columns or energy_df.index.dtype == 'datetime64[ns]':
            axes[0, 1].plot(energy_df.index[:1000], energy_df[target_col].iloc[:1000])
            axes[0, 1].set_xlabel('Время')
            axes[0, 1].set_ylabel('Потребление')
            axes[0, 1].set_title('Временной ряд энергопотребления')
            axes[0, 1].tick_params(axis='x', rotation=45)
        
        # График 3: Корреляционная матрица
        corr_cols = numeric_cols[:10]  # Первые 10 числовых колонок
        if len(corr_cols) > 1:
            corr_matrix = energy_df[corr_cols].corr()
            sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                        ax=axes[1, 0], center=0)
            axes[1, 0].set_title('Корреляционная матрица')
        
        # График 4: Box plot
        energy_df.boxplot(column=numeric_cols[:5], ax=axes[1, 1])
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].set_title('Распределение (box plot)')
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Нет числовых колонок для анализа")

## 5. Подготовка данных для обучения

In [ ]:
if not energy_df.empty:
    print("🔧 Подготовка данных для обучения...\n")
    
    # Выбор числовых колонок
    numeric_cols = energy_df.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(numeric_cols) < 2:
        print("⚠️  Недостаточно числовых колонок для обучения")
    else:
        # Целевая переменная (первая числовая колонка)
        target_col = numeric_cols[0]
        feature_cols = numeric_cols[1:min(5, len(numeric_cols))]  # Используем до 5 признаков
        
        print(f"Целевая переменная: {target_col}")
        print(f"Признаки: {feature_cols}")
        
        # Удаление пропусков
        df_clean = energy_df[[target_col] + feature_cols].dropna()
        
        if len(df_clean) == 0:
            print("⚠️  Нет данных после удаления пропусков")
        else:
            print(f"✓ Размер данных после очистки: {df_clean.shape}")
            
            # Разделение на признаки и целевую переменную
            X = df_clean[feature_cols]
            y = df_clean[target_col]
            
            # Разделение на train/test
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )
            
            print(f"\n📊 Разделение данных:")
            print(f"  Train: {X_train.shape[0]} записей")
            print(f"  Test: {X_test.shape[0]} записей")
            
            # Масштабирование
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            print("✓ Данные подготовлены")

## 6. Обучение модели

In [ ]:
if not energy_df.empty and len(numeric_cols) >= 2:
    print("\n🤖 Обучение моделей...\n")
    
    # Модель 1: Linear Regression
    print("1. Обучение Linear Regression...")
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    
    # Предсказания
    y_pred_lr = lr_model.predict(X_test_scaled)
    
    # Метрики
    mae_lr = mean_absolute_error(y_test, y_pred_lr)
    rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
    r2_lr = r2_score(y_test, y_pred_lr)
    
    print(f"  MAE:  {mae_lr:.4f}")
    print(f"  RMSE: {rmse_lr:.4f}")
    print(f"  R²:   {r2_lr:.4f}")
    
    # Модель 2: Random Forest
    print("\n2. Обучение Random Forest...")
    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train_scaled, y_train)
    
    # Предсказания
    y_pred_rf = rf_model.predict(X_test_scaled)
    
    # Метрики
    mae_rf = mean_absolute_error(y_test, y_pred_rf)
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    r2_rf = r2_score(y_test, y_pred_rf)
    
    print(f"  MAE:  {mae_rf:.4f}")
    print(f"  RMSE: {rmse_rf:.4f}")
    print(f"  R²:   {r2_rf:.4f}")
    
    # Сравнение моделей
    print("\n" + "=" * 60)
    print("📊 СРАВНЕНИЕ МОДЕЛЕЙ")
    print("=" * 60)
    
    comparison = pd.DataFrame({
        'Модель': ['Linear Regression', 'Random Forest'],
        'MAE': [mae_lr, mae_rf],
        'RMSE': [rmse_lr, rmse_rf],
        'R²': [r2_lr, r2_rf]
    })
    
    display(comparison.sort_values('R²', ascending=False))
    
    # Выбор лучшей модели
    best_model = 'Random Forest' if r2_rf > r2_lr else 'Linear Regression'
    print(f"\n✅ Лучшая модель: {best_model}")

## 7. Визуализация результатов

In [ ]:
if not energy_df.empty and len(numeric_cols) >= 2:
    print("📈 Визуализация результатов...\n")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # График 1: Прогноз vs Реальные данные (Linear Regression)
    axes[0, 0].scatter(y_test.iloc[:100], y_pred_lr[:100], alpha=0.5, edgecolors='k', linewidth=0.5)
    axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[0, 0].set_xlabel('Реальные значения')
    axes[0, 0].set_ylabel('Прогноз')
    axes[0, 0].set_title(f'Linear Regression\nR² = {r2_lr:.4f}')
    axes[0, 0].grid(True, alpha=0.3)
    
    # График 2: Прогноз vs Реальные данные (Random Forest)
    axes[0, 1].scatter(y_test.iloc[:100], y_pred_rf[:100], alpha=0.5, edgecolors='k', linewidth=0.5)
    axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[0, 1].set_xlabel('Реальные значения')
    axes[0, 1].set_ylabel('Прогноз')
    axes[0, 1].set_title(f'Random Forest\nR² = {r2_rf:.4f}')
    axes[0, 1].grid(True, alpha=0.3)
    
    # График 3: Ошибки прогноза
    errors_lr = np.abs(y_test.iloc[:100] - y_pred_lr[:100])
    errors_rf = np.abs(y_test.iloc[:100] - y_pred_rf[:100])
    
    axes[1, 0].hist(errors_lr, bins=30, alpha=0.7, label='Linear Regression', edgecolor='black')
    axes[1, 0].hist(errors_rf, bins=30, alpha=0.7, label='Random Forest', edgecolor='black')
    axes[1, 0].set_xlabel('Абсолютная ошибка')
    axes[1, 0].set_ylabel('Частота')
    axes[1, 0].set_title('Распределение ошибок')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # График 4: Важность признаков (Random Forest)
    if hasattr(rf_model, 'feature_importances_'):
        importance = pd.DataFrame({
            'Признак': feature_cols,
            'Важность': rf_model.feature_importances_
        }).sort_values('Важность', ascending=True)
        
        axes[1, 1].barh(importance['Признак'], importance['Важность'])
        axes[1, 1].set_xlabel('Важность')
        axes[1, 1].set_title('Важность признаков (Random Forest)')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Временной ряд прогноза
    if 'timestamp' in energy_df.columns or energy_df.index.dtype == 'datetime64[ns]':
        plt.figure(figsize=(16, 6))
        
        # Индекс для тестовой выборки
        test_idx = y_test.index[:200]
        
        plt.plot(test_idx, y_test.iloc[:200], label='Реальные', linewidth=2)
        plt.plot(test_idx, y_pred_rf[:200], label='Прогноз (RF)', linestyle='--', linewidth=2)
        
        plt.xlabel('Время')
        plt.ylabel('Потребление')
        plt.title('Прогноз энергопотребления во времени')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 8. Сохранение модели

In [ ]:
if not energy_df.empty and len(numeric_cols) >= 2:
    print("💾 Сохранение модели...\n")
    
    # Выбор лучшей модели для сохранения
    if r2_rf >= r2_lr:
        best_model_obj = rf_model
        model_name = 'random_forest_energy'
        print(f"Сохранение Random Forest...")
    else:
        best_model_obj = lr_model
        model_name = 'linear_regression_energy'
        print(f"Сохранение Linear Regression...")
    
    # Сохранение модели
    model_path = MODELS_DIR / f'{model_name}.pkl'
    joblib.dump(best_model_obj, model_path)
    
    # Сохранение scaler
    scaler_path = MODELS_DIR / 'scaler.pkl'
    joblib.dump(scaler, scaler_path)
    
    # Сохранение метрик
    metrics = {
        'model': model_name,
        'timestamp': datetime.now().isoformat(),
        'features': feature_cols,
        'target': target_col,
        'metrics': {
            'Linear Regression': {
                'MAE': float(mae_lr),
                'RMSE': float(rmse_lr),
                'R2': float(r2_lr)
            },
            'Random Forest': {
                'MAE': float(mae_rf),
                'RMSE': float(rmse_rf),
                'R2': float(r2_rf)
            }
        }
    }
    
    metrics_path = MODELS_DIR / 'energy_metrics.json'
    with open(metrics_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ Модель сохранена: {model_path}")
    print(f"✓ Scaler сохранён: {scaler_path}")
    print(f"✓ Метрики сохранены: {metrics_path}")

## 9. Сводка результатов

In [ ]:
from IPython.display import display, Markdown

if not energy_df.empty and len(numeric_cols) >= 2:
    display(Markdown("""
## 📋 Сводка

### Выполненные действия:
1. ✅ Загружены данные BDGP2 (Building Data Genome Project 2)
2. ✅ Проведён исследовательский анализ данных (EDA)
3. ✅ Подготовлены данные для обучения (масштабирование, train/test split)
4. ✅ Обучены две модели: Linear Regression и Random Forest
5. ✅ Рассчитаны метрики: MAE, RMSE, R²
6. ✅ Визуализированы результаты (прогноз vs реальные данные)
7. ✅ Сохранена лучшая модель в `models/predictive/`

### Метрики лучшей модели:
    """))
    
    if r2_rf >= r2_lr:
        print(f"🏆 Модель: Random Forest")
        print(f"   MAE:  {mae_rf:.4f}")
        print(f"   RMSE: {rmse_rf:.4f}")
        print(f"   R²:   {r2_rf:.4f}")
    else:
        print(f"🏆 Модель: Linear Regression")
        print(f"   MAE:  {mae_lr:.4f}")
        print(f"   RMSE: {rmse_lr:.4f}")
        print(f"   R²:   {r2_lr:.4f}")
    
    print(f"\n📅 Дата выполнения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"📁 Модель сохранена: {MODELS_DIR}")
    
    display(Markdown("""
### Следующие шаги:
- Интеграция модели в API предиктивной аналитики
- Добавление прогнозирования аномалий
- Развёртывание в продакшн
"""))
else:
    display(Markdown("""
## ⚠️ Данные не загружены

Запустите скачивание датасетов:
```bash
python scripts/download_datasets.py
```

Или поместите данные BDGP2 вручную в:
`datasets/energy/data/bdgp2/`
"""))